In [0]:
env = dbutils.widgets.get("environment") if dbutils.widgets.getAll().get("environment") else "dev"
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawRegion"
silverTablName = f"saleslake_{env}.silver_{env}.cleanedRegion"

spark.sql(f"""
INSERT INTO {silverTablName}
SELECT DISTINCT
    CAST(TRIM(region_id) AS INT)              AS region_id,
    UPPER(TRIM(region_code))                  AS region_code,
    UPPER(TRIM(region_name))                  AS region_name,
    UPPER(TRIM(country))                      AS country,
    UPPER(TRIM(sub_region))                   AS sub_region,
    TRIM(regional_manager)                    AS regional_manager,
    TO_DATE(TRIM(created_date), 'yyyy-MM-dd') AS created_date,
    CURRENT_TIMESTAMP() AS ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP("1990-01-01"))
    FROM {silverTablName}
)
ORDER BY region_id
""")

print(f"Silver load complete for {silverTablName}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {silverTablName}").display()